In [3]:
from langchain_ollama import OllamaEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from langchain_community.document_loaders import PyPDFLoader

# Embedding Models — Ollama (Local, Free, Private)

## What this notebook demonstrates

This notebook shows how to use **Ollama's embedding models** to build a mini RAG pipeline:

1. **Create documents** — financial statement text split into structured chunks
2. **Split into chunks** — use `RecursiveCharacterTextSplitter` to break documents into smaller pieces
3. **Embed the chunks** — convert each chunk to a vector using Ollama's `nomic-embed-text` model
4. **Embed a query** — convert a question to the same vector space
5. **Compare** — find which chunk is most relevant to the question using cosine similarity

## Why chunk documents before embedding?

Embedding models have a **context limit** (maximum text length they can process at once). Long documents must be split into smaller **chunks** first.

But there's a deeper reason: smaller chunks give more precise retrieval. If your document is 50 pages and the answer is on page 12, you want to retrieve just page 12's content — not all 50 pages. Chunking makes retrieval more accurate.

```
Full document (too long to embed as one piece)
    ↓ RecursiveCharacterTextSplitter
Chunk 1 (page 1-2)   → vector_1
Chunk 2 (page 3-4)   → vector_2   ← stored in vector DB
Chunk 3 (page 5-6)   → vector_3
    ↓ Query time
User question → query_vector → compare to all chunk vectors → return most relevant chunk
```

## `nomic-embed-text` — the model used here

A high-quality open-source embedding model available through Ollama:
- Pull it: `ollama pull nomic-embed-text`
- Strong performance, fully local, zero cost

## Prerequisites

- Ollama running + `ollama pull nomic-embed-text`
- Virtual environment activated

In [ ]:
from langchain.schema import Document

# Using inline documents so this notebook works without any external files.
# In a real RAG pipeline you would load these from PDFs, web pages, databases, etc.
docs = [
    Document(
        page_content=(
            "Sample Company — Income Statement (Service)\n"
            "For the Year Ended September 30, 2021\n\n"
            "Service revenue: $2,750\n"
            "Operating Expenses:\n"
            "  Depreciation: $100 | Wages: $1,200 | Supplies: $60\n"
            "  Total operating expenses: $1,360\n"
            "Operating Income: $1,390\n"
            "Interest expense: $40 | Pretax income: $1,350\n"
            "Income tax expense: $405 | Net income: $945\n\n"
            "Statement of Retained Earnings:\n"
            "  Balance Oct 1 2020: $820 | Net income: $945 | Dividends: -$500\n"
            "  Balance Sep 30 2021: $1,265"
        ),
        metadata={"source": "income_statement_service", "page": 1},
    ),
    Document(
        page_content=(
            "Sample Company — Income Statement (Product)\n"
            "For the Year Ended September 30, 2021\n\n"
            "Sales revenue: $6,875 | Cost of Goods Sold: -$4,125\n"
            "Gross Profit: $2,750\n"
            "Operating Expenses:\n"
            "  Depreciation: $100 | Wages: $1,200 | Supplies: $60\n"
            "  Total operating expenses: $1,360\n"
            "Operating Income: $1,390\n"
            "Interest expense: $40 | Pretax income: $1,350\n"
            "Income tax expense: $405 | Net income: $945"
        ),
        metadata={"source": "income_statement_product", "page": 2},
    ),
    Document(
        page_content=(
            "Sample Company — Balance Sheet\n"
            "September 30, 2021\n\n"
            "ASSETS:\n"
            "  Cash: $1,550 | Accounts receivable: $770 | Supplies: $40\n"
            "  Equipment: $12,000 | Accumulated depreciation: -$1,300\n"
            "  Total assets: $13,060\n\n"
            "LIABILITIES:\n"
            "  Accounts payable: $60 | Interest payable: $80\n"
            "  Wages payable: $100 | Income taxes payable: $405\n"
            "  Utilities payable: $250 | Total current liabilities: $895\n"
            "  Long-term notes payable: $8,000\n\n"
            "OWNERS' EQUITY:\n"
            "  Owners' capital: $2,900 | Retained earnings: $1,265\n"
            "  Total equity: $4,165\n"
            "  Total liabilities + equity: $13,060"
        ),
        metadata={"source": "balance_sheet", "page": 3},
    ),
]

print(f"Loaded {len(docs)} documents")

In [16]:
docs

[Document(metadata={'producer': 'Adobe PDF Library 21.7.131', 'creator': 'Acrobat PDFMaker 21 for Word', 'creationdate': '2021-12-08T11:03:22-05:00', 'author': 'Kathryn Jervis', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2021-12-08T11:03:56-05:00', 'sourcemodified': '', 'subject': '', 'title': '', 'source': '/Users/rahuljauhari/Downloads/Sample-Financial-Statements-1.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='Sample Company \nIncome Statement (Service) \nFor the Year Ended September 30, 2021 \n  \nService revenue  $2,750 \n \nOperating Expenses:  \n Depreciation expense    100 \n Wages expenses 1,200 \n Supplies expenses      60 \nTotal operating expenses   1,360 \n \nOperating Income   1,390 \n \nOther Item:  \n Interest expense      40 \nPretax income  1,350 \n \n Income tax expense    405 \nNet income   $945 \n \n \n \n \n \nSample Company \nStatement of Retained Earnings \nFor the Year Ended September 30, 2021 \n \n \n    Retained Earnings 

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs)

In [ ]:
# nomic-embed-text is a high-quality open-source embedding model available via Ollama.
# Pull it first if you haven't: ollama pull nomic-embed-text
embedding_model = OllamaEmbeddings(model="nomic-embed-text")

In [24]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 21.7.131', 'creator': 'Acrobat PDFMaker 21 for Word', 'creationdate': '2021-12-08T11:03:22-05:00', 'author': 'Kathryn Jervis', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2021-12-08T11:03:56-05:00', 'sourcemodified': '', 'subject': '', 'title': '', 'source': '/Users/rahuljauhari/Downloads/Sample-Financial-Statements-1.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='Sample Company \nIncome Statement (Service) \nFor the Year Ended September 30, 2021 \n  \nService revenue  $2,750 \n \nOperating Expenses:  \n Depreciation expense    100 \n Wages expenses 1,200 \n Supplies expenses      60 \nTotal operating expenses   1,360 \n \nOperating Income   1,390 \n \nOther Item:  \n Interest expense      40 \nPretax income  1,350 \n \n Income tax expense    405 \nNet income   $945 \n \n \n \n \n \nSample Company \nStatement of Retained Earnings \nFor the Year Ended September 30, 2021 \n \n \n    Retained Earnings 

In [ ]:
# embed_documents() expects a list of plain strings, NOT Document objects.
# We extract .page_content from each chunk before passing to the embedding model.
document_embeddings = embedding_model.embed_documents([c.page_content for c in chunks])
print(f"Generated {len(document_embeddings)} embeddings")
print(f"Each embedding has {len(document_embeddings[0])} dimensions")

In [7]:
document_embeddings[0]

[-0.021085924,
 0.03824834,
 0.01782326,
 -0.0038380357,
 0.014819041,
 0.011859054,
 -0.030068485,
 0.06872403,
 0.055070225,
 -0.016367301,
 0.005732714,
 0.03381014,
 -0.017121512,
 -0.02024273,
 0.008354044,
 -0.0109846145,
 -0.010049647,
 0.019138377,
 0.024472648,
 -0.0044507724,
 0.013363115,
 0.005358458,
 -0.046233907,
 -0.024835836,
 0.03344004,
 0.0259435,
 0.013282848,
 -0.014161466,
 0.036346167,
 0.041985024,
 0.027389241,
 -0.035170827,
 0.008824573,
 -0.04548043,
 -0.039040394,
 0.020907471,
 0.019910913,
 -0.00076070474,
 0.014596954,
 -0.026848203,
 0.024805387,
 -0.0516569,
 0.011345991,
 -0.08185969,
 -0.0037343872,
 -0.08129282,
 -0.040223654,
 0.007374311,
 0.05744302,
 0.0016974913,
 -0.022390343,
 0.015321481,
 0.04407464,
 -0.019760521,
 -0.040993914,
 0.0008466813,
 -0.035066523,
 0.026755733,
 0.01871479,
 0.029790642,
 0.008509169,
 -0.0037523143,
 -0.00028863532,
 -0.037920885,
 0.010991479,
 0.044363957,
 0.03218104,
 -0.021154327,
 0.0064157876,
 -0.02928

In [8]:
enum = enumerate(document_embeddings)

In [9]:
for idx, val in enum:
    print(idx, val)

0 [-0.021085924, 0.03824834, 0.01782326, -0.0038380357, 0.014819041, 0.011859054, -0.030068485, 0.06872403, 0.055070225, -0.016367301, 0.005732714, 0.03381014, -0.017121512, -0.02024273, 0.008354044, -0.0109846145, -0.010049647, 0.019138377, 0.024472648, -0.0044507724, 0.013363115, 0.005358458, -0.046233907, -0.024835836, 0.03344004, 0.0259435, 0.013282848, -0.014161466, 0.036346167, 0.041985024, 0.027389241, -0.035170827, 0.008824573, -0.04548043, -0.039040394, 0.020907471, 0.019910913, -0.00076070474, 0.014596954, -0.026848203, 0.024805387, -0.0516569, 0.011345991, -0.08185969, -0.0037343872, -0.08129282, -0.040223654, 0.007374311, 0.05744302, 0.0016974913, -0.022390343, 0.015321481, 0.04407464, -0.019760521, -0.040993914, 0.0008466813, -0.035066523, 0.026755733, 0.01871479, 0.029790642, 0.008509169, -0.0037523143, -0.00028863532, -0.037920885, 0.010991479, 0.044363957, 0.03218104, -0.021154327, 0.0064157876, -0.029280607, -0.01928547, 0.057130214, -0.044191215, 0.009437547, -0.05236

In [10]:
query = embedding_model.embed_query("What is the capital of Germany?")
query

[-0.02706326,
 -0.013210408,
 0.016376289,
 -0.0054852096,
 0.011527921,
 0.004724984,
 0.011158029,
 -0.009387243,
 -0.045015346,
 -0.038716294,
 0.008081667,
 0.00090078893,
 -0.018211134,
 0.017297387,
 0.012351493,
 -0.0060215616,
 -0.011829639,
 -0.0023606094,
 -0.030375881,
 0.011038896,
 0.00013819299,
 -0.018009843,
 -0.025188701,
 -0.03206448,
 0.013591763,
 0.05960964,
 0.01923585,
 0.0042280476,
 -0.007707877,
 0.029691467,
 -0.010382375,
 -0.06554636,
 0.011841922,
 0.010718545,
 -0.0341979,
 0.006730495,
 0.01884743,
 -0.05329378,
 0.025561756,
 -0.083847806,
 -0.001799413,
 -0.041181043,
 -0.0028906956,
 -0.03286592,
 -0.021771263,
 0.016377103,
 -0.037538774,
 0.00555479,
 0.00093347474,
 -0.012654482,
 0.015102538,
 0.0137419505,
 -0.03798292,
 0.020270444,
 -0.016455133,
 -0.06731435,
 0.013589471,
 0.007831315,
 -0.012137959,
 0.011656673,
 0.0040378883,
 -0.009562254,
 0.015301949,
 -0.011462938,
 0.018003384,
 -0.0011393211,
 0.03546986,
 -0.063649245,
 -0.006636801

In [11]:
result_enum = cosine_similarity([query], document_embeddings)
result_enum

array([[0.2616038, 0.2034764]])

In [12]:
similarity_scores = result_enum[0]
similarity_scores

array([0.2616038, 0.2034764])

In [13]:
max_index = np.argmax(similarity_scores)
max_index

np.int64(0)

In [14]:
highest_similar_document = documents[max_index]
highest_similar_document

'The cat is on the roof.'

In [15]:
print("Question:",query)
print("Most similar document:", highest_similar_document)
print("Similarity score:", (similarity_scores[max_index]/1.0)*100, "%")

Question: [-0.02706326, -0.013210408, 0.016376289, -0.0054852096, 0.011527921, 0.004724984, 0.011158029, -0.009387243, -0.045015346, -0.038716294, 0.008081667, 0.00090078893, -0.018211134, 0.017297387, 0.012351493, -0.0060215616, -0.011829639, -0.0023606094, -0.030375881, 0.011038896, 0.00013819299, -0.018009843, -0.025188701, -0.03206448, 0.013591763, 0.05960964, 0.01923585, 0.0042280476, -0.007707877, 0.029691467, -0.010382375, -0.06554636, 0.011841922, 0.010718545, -0.0341979, 0.006730495, 0.01884743, -0.05329378, 0.025561756, -0.083847806, -0.001799413, -0.041181043, -0.0028906956, -0.03286592, -0.021771263, 0.016377103, -0.037538774, 0.00555479, 0.00093347474, -0.012654482, 0.015102538, 0.0137419505, -0.03798292, 0.020270444, -0.016455133, -0.06731435, 0.013589471, 0.007831315, -0.012137959, 0.011656673, 0.0040378883, -0.009562254, 0.015301949, -0.011462938, 0.018003384, -0.0011393211, 0.03546986, -0.063649245, -0.0066368016, 0.0014976104, -0.011037255, -0.022075372, 0.025700787, 